### You will notice in crew you don't do load_dotenv, that is because crew is already doing it

# How the Latest my_stock_picker Works

## Overview: Dual-Perspective Stock Analysis System

The latest my_stock_picker implementation uses **two separate finder agents** (Grok and Gemini) to get diverse stock perspectives, then consolidates research through a single researcher and rater.

## Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────┐
│                    Manager (Gemini)                             │
│                   Coordinates Workflow                          │
└───────┬─────────────────────────────────────────────────────────┘
        │
        ├─────────────────────────────────────────────────────────┐
        │                                                         │
        ▼                                                         ▼
┌───────────────────────┐                         ┌───────────────────────┐
│  Finder Agent #1      │                         │  Finder Agent #2      │
│  (Grok-powered)       │                         │  (Gemini-powered)     │
│                       │                         │                       │
│  Focus: Fundamentals  │                         │  Focus: Momentum      │
│  Long-term trends     │                         │  Short-term catalysts │
└───────┬───────────────┘                         └───────┬───────────────┘
        │                                                 │
        │ Outputs 2-3 stocks                             │ Outputs 2-3 stocks
        │ (fundamental focus)                            │ (momentum focus)
        │                                                 │
        └─────────────────┬───────────────────────────────┘
                          │
                          ▼
                ┌─────────────────────┐
                │  Financial          │
                │  Researcher (Grok)  │
                │                     │
                │  Receives BOTH      │
                │  lists, researches  │
                │  all unique stocks  │
                └──────────┬──────────┘
                           │
                           │ Comprehensive research
                           │
                           ▼
                  ┌────────────────┐
                  │  Stock Rater   │
                  │  (Grok)        │
                  │                │
                  │  Rates 1-100   │
                  │  All stocks    │
                  └────────────────┘
```

## Agent Breakdown

### 1. Manager Agent (Gemini)
- **Role**: Coordinates the entire workflow
- **LLM**: Gemini 2.5 Flash (fast, efficient coordination)
- **Responsibilities**:
  - Delegates tasks to appropriate agents
  - Ensures workflow completion
  - In hierarchical mode, decides which agent works on which task

### 2. Finder Agent #1: Grok Perspective
- **Name**: `trending_company_finder_grok`
- **LLM**: Grok 4 Fast (thorough analysis)
- **Focus**: Fundamental analysis and long-term trends
- **Personality**: Methodical, thorough, seasoned market expert
- **Output**: 2-3 publicly traded companies with strong fundamentals
- **Output File**: `output/trending_companies_grok.json`

**What makes it unique:**
- Looks for companies with strong fundamentals
- Focuses on long-term value and trends
- Takes a more conservative, analytical approach

### 3. Finder Agent #2: Gemini Perspective
- **Name**: `trending_company_finder_gemini`
- **LLM**: Gemini 2.5 Flash (fast, momentum-focused)
- **Focus**: Momentum and short-term catalysts
- **Personality**: Quick-thinking, spots fast-moving trends
- **Output**: 2-3 publicly traded companies with strong momentum
- **Output File**: `output/trending_companies_gemini.json`
- **Special Instruction**: Try to find DIFFERENT companies than Grok finder

**What makes it unique:**
- Looks for momentum plays
- Focuses on short-term catalysts and news
- Takes a more aggressive, trend-following approach

### 4. Financial Researcher (Grok)
- **Name**: `financial_researcher`
- **LLM**: Grok 4 Fast (thorough research)
- **Input**: BOTH lists from the two finders
- **Responsibilities**:
  - Combine companies from both lists
  - Remove duplicates
  - Research each unique company thoroughly
- **Output**: Comprehensive research report
- **Output File**: `output/research_report.json`

### 5. Stock Rater (Grok)
- **Name**: `stock_rater`
- **LLM**: Grok 4 Fast (consistent rating)
- **Input**: Research from financial_researcher
- **Responsibilities**:
  - Rate each company on a scale of 1-100
  - Provide reasoning for each rating
  - Rank companies highest to lowest
- **Output**: Ranked list with ratings and reasons
- **Output File**: `output/decision.md`

## Task Flow

### Task 1: find_trending_companies_grok
```yaml
description: Find PUBLICLY TRADED companies with focus on fundamentals
agent: trending_company_finder_grok
output: TrendingCompanies (Pydantic model)
```

### Task 2: find_trending_companies_gemini
```yaml
description: Find PUBLICLY TRADED companies with focus on momentum
agent: trending_company_finder_gemini
output: TrendingCompanies (Pydantic model)
```

### Task 3: research_trending_companies
```yaml
description: Research ALL companies from BOTH lists
agent: financial_researcher
context:
  - find_trending_companies_grok
  - find_trending_companies_gemini
output: TrendingCompanyResearchList (Pydantic model)
```

### Task 4: rate_trending_companies
```yaml
description: Rate all researched companies 1-100
agent: stock_rater
context:
  - research_trending_companies
output: Markdown report
```

## Execution Flow

1. **Manager assigns Task 1** → Grok Finder searches for fundamental plays
   - Example output: NVDA, MSFT, GOOGL (strong fundamentals)

2. **Manager assigns Task 2** → Gemini Finder searches for momentum plays
   - Example output: SMCI, PLTR, SNOW (high momentum)

3. **Manager assigns Task 3** → Researcher receives BOTH lists
   - Combined list: NVDA, MSFT, GOOGL, SMCI, PLTR, SNOW
   - Removes duplicates if any overlap
   - Researches all 6 companies thoroughly

4. **Manager assigns Task 4** → Rater analyzes research
   - Rates each of the 6 companies
   - Example output:
     ```
     1. NVDA - 95/100 (Strong fundamentals + momentum)
     2. MSFT - 88/100 (Solid fundamentals)
     3. PLTR - 82/100 (High momentum)
     ...
     ```

## Key Benefits of This Design

### 1. Diverse Perspectives
- **Grok Finder**: Value/fundamental investing approach
- **Gemini Finder**: Growth/momentum investing approach
- **Result**: Covers both investment styles

### 2. Comprehensive Coverage
- Get 4-6 unique companies instead of just 2-3
- Mix of stable fundamentals and high-growth opportunities
- Reduces risk of missing good opportunities

### 3. Consistent Analysis
- **Single Researcher**: All companies analyzed with same methodology
- **Single Rater**: All companies rated with same criteria
- **Result**: Fair, comparable ratings

### 4. LLM Optimization
- **Gemini** for fast tasks (manager, momentum scanning)
- **Grok** for thorough tasks (fundamental analysis, research, rating)
- **Result**: Balance of speed and quality

## Pydantic Models

### TrendingCompany
```python
class TrendingCompany(BaseModel):
    name: str                      # Company name
    ticker: str                    # Stock ticker (REQUIRED - public only)
    price: float | None            # Current price (optional)
    projected_price: float | None  # 5-year projection (optional)
    reason: str                    # Why it's trending
```

### TrendingCompanies
```python
class TrendingCompanies(BaseModel):
    companies: list[TrendingCompany]  # List of 2-3 companies
```

### TrendingCompanyResearch
```python
class TrendingCompanyResearch(BaseModel):
    name: str                  # Company name
    market_position: str       # Market analysis
    future_outlook: str        # Growth prospects
    investment_potential: str  # Investment analysis
```

## Important Constraints

### Only Publicly Traded Companies
Both finders are instructed to:
- **ONLY** include companies with stock tickers
- **EXCLUDE** private companies (Perplexity, Cohere, etc.)
- **EXCLUDE** startups without tickers

This ensures all results are actually investable.

## Output Files

1. **trending_companies_grok.json**: Grok's fundamental picks
2. **trending_companies_gemini.json**: Gemini's momentum picks
3. **research_report.json**: Combined research on all companies
4. **decision.md**: Final ratings and recommendations

## Running the Crew

### Basic Run
```bash
cd ~/projects/agents/3_crew/my_stock_picker
crewai run
# Prompts for sector input (e.g., "technology")
```

### With Fallback
```bash
python run_with_diverse_opinions.py
# Automatically handles LLM failures with fallback chain
```

## LLM Fallback Strategy

The crew supports fallback configurations:

```python
llm_configs = [
    {  # Ideal: Diverse perspectives
        'manager': 'gemini-2.5-flash',
        'researcher': 'grok-4-fast',
        'analyst': 'gemini-2.5-flash',
        'rater': 'grok-4-fast'
    },
    {  # Fallback: All Gemini if Grok unavailable
        'manager': 'gemini-2.5-flash',
        'researcher': 'gemini-2.5-flash',
        'analyst': 'gemini-2.5-flash',
        'rater': 'gemini-2.5-flash'
    },
    {  # Final: All local Ollama
        'manager': 'ollama-gemma27b',
        'researcher': 'ollama-gemma27b',
        'analyst': 'ollama-gemma27b',
        'rater': 'ollama-gemma27b'
    }
]
```

## Comparison: Old vs New Design

| Aspect | Old Design | New Design |
|--------|-----------|------------|
| **Finders** | 1 agent | 2 agents (Grok + Gemini) |
| **Perspectives** | Single view | Dual perspectives (fundamental + momentum) |
| **Stock Coverage** | 2-3 companies | 4-6 companies |
| **Diversity** | Limited | High (two different "minds") |
| **Research** | Single list | Combined from two lists |
| **Rating** | Same | Same (consistent) |

## Why This Works Well

1. **Two Finders = Two Investment Styles**
   - Grok: Conservative, fundamental (like Warren Buffett)
   - Gemini: Aggressive, momentum (like growth traders)

2. **Single Research/Rating = Consistency**
   - All companies analyzed with same rigor
   - Fair comparison across different types of stocks

3. **Better Portfolio Balance**
   - Mix of stable (fundamental) and high-growth (momentum)
   - Reduces portfolio risk through diversification

4. **Leverages LLM Strengths**
   - Gemini: Fast scanning and coordination
   - Grok: Deep analysis and reasoning

# CrewAI: 5 Basic Steps to Build a Crew

## Overview

CrewAI is a framework for orchestrating multiple AI agents to work together on complex tasks. Here are the 5 essential steps to create and run a crew project.

---

## Step 1: Create a New Crew Project

Use the CrewAI CLI to scaffold a new project:

```bash
crewai create crew my_crew
```

This creates:
```
my_crew/
├── src/
│   └── my_crew/
│       ├── config/
│       │   ├── agents.yaml    # Agent definitions
│       │   └── tasks.yaml     # Task definitions
│       ├── crew.py            # Main crew implementation
│       ├── main.py            # Entry point
│       └── tools/             # Custom tools (optional)
├── pyproject.toml             # Dependencies
└── README.md
```

---

## Step 2: Configure Agents and Tasks

### Edit `config/agents.yaml`

Define your agents with roles, goals, and backstories:

```yaml
researcher:
  role: Senior Research Analyst
  goal: Find and analyze trending companies in {sector}
  backstory: >
    You're an experienced financial analyst with 15 years of experience.
    You excel at identifying market trends and investment opportunities.

analyst:
  role: Investment Analyst
  goal: Evaluate companies for investment potential
  backstory: >
    You're a detail-oriented analyst who excels at financial modeling
    and risk assessment.
```

### Edit `config/tasks.yaml`

Define tasks with descriptions and expected outputs:

```yaml
research_task:
  description: >
    Research trending companies in the {sector} sector.
    Find at least 3 publicly traded companies with strong momentum.
  expected_output: >
    A JSON list of companies with ticker symbols, current price,
    and reason for being trending.
  agent: researcher

analysis_task:
  description: >
    Analyze the companies from the research task.
    Rate each company's investment potential on a 1-100 scale.
  expected_output: >
    A ranked list with ratings and detailed justifications.
  agent: analyst
  context:
    - research_task  # Uses output from research_task
```

---

## Step 3: Implement `crew.py`

The `crew.py` file ties everything together with LLMs, agents, tasks, and crew configuration.

### Basic Structure

```python
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool

# Import your custom LLM configuration
import sys
from pathlib import Path
sys.path.append(str(Path(__file__).parent.parent.parent.parent))
from shared_llms import build_llms

@CrewBase
class MyCrew():
    """MyCrew crew implementation"""
    
    agents_config = 'config/agents.yaml'
    tasks_config = 'config/tasks.yaml'

    def __init__(self):
        # Build LLMs
        llms = build_llms(['grok-4-fast', 'gemini-2.5-flash', 'ollama-gemma27b'])
        self.llm_grok = llms['grok-4-fast']
        self.llm_gemini = llms['gemini-2.5-flash']
        self.llm_ollama = llms['ollama-gemma27b']

    @agent
    def researcher(self) -> Agent:
        """Define the researcher agent"""
        return Agent(
            config=self.agents_config['researcher'],
            llm=self.llm_grok,  # Assign specific LLM
            tools=[SerperDevTool()],  # Web search tool
            max_rpm=10,  # Rate limit: 10 requests per minute
            verbose=True
        )

    @agent
    def analyst(self) -> Agent:
        """Define the analyst agent"""
        return Agent(
            config=self.agents_config['analyst'],
            llm=self.llm_gemini,  # Different LLM for diversity
            max_rpm=8,  # Conservative limit for free tier
            verbose=True
        )

    @task
    def research_task(self) -> Task:
        """Research trending companies"""
        return Task(
            config=self.tasks_config['research_task'],
        )

    @task
    def analysis_task(self) -> Task:
        """Analyze companies"""
        return Task(
            config=self.tasks_config['analysis_task'],
        )

    @crew
    def crew(self) -> Crew:
        """Creates the crew"""
        return Crew(
            agents=self.agents,  # Auto-populated from @agent decorators
            tasks=self.tasks,    # Auto-populated from @task decorators
            process=Process.sequential,  # Or Process.hierarchical
            verbose=True,
        )
```

### Rate Limiting with max_rpm

CrewAI provides built-in rate limiting through the `max_rpm` parameter:

```python
@agent
def my_agent(self) -> Agent:
    return Agent(
        config=self.agents_config['my_agent'],
        llm=self.llm_gemini,
        max_rpm=8,  # Limit to 8 requests per minute
        verbose=True
    )
```

**Why use max_rpm?**
- Prevents 429 errors (rate limit exceeded)
- Required for free tiers:
  - Gemini 2.5 Flash: 10 req/min (use 8 for safety)
  - Gemini 2.5 Pro: 2 req/min
  - Grok: Varies by account (usually 10 req/min)
- Local models (Ollama): No limit needed

**Per-agent vs global:**
- `max_rpm` is per-agent, not global
- Each agent gets its own rate limit
- Total crew rate = sum of all agents' limits

**Conditional rate limiting:**
```python
@agent
def my_agent(self) -> Agent:
    return Agent(
        config=self.agents_config['my_agent'],
        llm=self.llm_gemini,
        # Only limit Gemini, not Ollama
        max_rpm=8 if self.llm_gemini else None,
        verbose=True
    )
```

See [USING_MAX_RPM.md](../USING_MAX_RPM.md) for comprehensive rate limiting guide.

### Sequential vs Hierarchical Process

#### Sequential (Simple)
Tasks run one after another in order:

```python
process=Process.sequential
```

**Use when:**
- Linear workflow (A → B → C)
- Tasks have clear dependencies
- Simpler coordination

#### Hierarchical (Advanced)
A manager agent coordinates and delegates tasks:

```python
# Create manager agent
manager = Agent(
    config=self.agents_config['manager'],
    llm=self.llm_gemini,
    allow_delegation=True,  # Required!
    max_rpm=8,  # Manager also needs rate limiting
    verbose=True
)

return Crew(
    agents=self.agents,
    tasks=self.tasks,
    process=Process.hierarchical,
    manager_agent=manager,  # Or use manager_llm
    verbose=True,
)
```

**Use when:**
- Complex coordination needed
- Manager should decide task assignment
- Parallel work possible

---

## Step 4: Configure `main.py`

The entry point handles inputs and kicks off the crew:

```python
#!/usr/bin/env python
import sys
import warnings
from my_crew.crew import MyCrew

warnings.filterwarnings("ignore", category=SyntaxWarning, module="pysbd")

def run():
    """
    Run the crew with user inputs.
    """
    inputs = {
        'sector': 'technology'  # Can be made interactive
    }
    
    # Alternative: Get input from user
    # inputs = {'sector': input("Enter sector: ")}
    
    MyCrew().crew().kickoff(inputs=inputs)

def train():
    """Train the crew for optimization"""
    inputs = {'sector': 'technology'}
    try:
        MyCrew().crew().train(
            n_iterations=int(sys.argv[1]),
            filename=sys.argv[2],
            inputs=inputs
        )
    except Exception as e:
        raise Exception(f"Training error: {e}")

def replay():
    """Replay a specific task"""
    try:
        MyCrew().crew().replay(task_id=sys.argv[1])
    except Exception as e:
        raise Exception(f"Replay error: {e}")

def test():
    """Test the crew"""
    inputs = {'sector': 'technology'}
    try:
        MyCrew().crew().test(
            n_iterations=int(sys.argv[1]),
            openai_model_name=sys.argv[2],
            inputs=inputs
        )
    except Exception as e:
        raise Exception(f"Testing error: {e}")
```

### Multi-LLM Setup

To use multiple LLMs, use `shared_llms.py`:

```python
# In crew.py __init__
llms = build_llms(['grok-4-fast', 'gemini-2.5-flash', 'ollama-gemma27b'])
self.llm_grok = llms['grok-4-fast']
self.llm_gemini = llms['gemini-2.5-flash']
self.llm_ollama = llms['ollama-gemma27b']
```

**LLM Selection Strategy:**
- **Gemini**: Fast coordination and quick tasks
- **Grok**: Thorough research and deep analysis
- **Ollama**: Local model, unlimited requests, good for testing

---

## Step 5: Run Your Crew

From the project root directory (where `pyproject.toml` is):

```bash
crewai run
```

**Important:**
- Must run from project root (not `src/` subdirectory)
- Command is `crewai run`, not `crew run`
- Activates virtual environment automatically

### Alternative: Run with Python

```bash
python src/my_crew/main.py
```

### Run with Custom Script

For advanced features like fallback:

```bash
python run_with_fallback.py
```

---

## Quick Reference

| Step | Command | File |
|------|---------|------|
| 1. Create | `crewai create crew my_crew` | - |
| 2. Configure | Edit YAML files | `config/agents.yaml`, `config/tasks.yaml` |
| 3. Implement | Write crew logic | `crew.py` |
| 4. Entry point | Configure inputs | `main.py` |
| 5. Run | `crewai run` | - |

---

## Common Patterns

### Multiple LLMs (Diverse Opinions)
```python
@agent
def researcher(self) -> Agent:
    return Agent(config=..., llm=self.llm_grok, max_rpm=10)  # Thorough

@agent
def analyst(self) -> Agent:
    return Agent(config=..., llm=self.llm_gemini, max_rpm=8)  # Fast
```

### Pydantic Output Models
```python
from pydantic import BaseModel

class Company(BaseModel):
    name: str
    ticker: str
    reason: str

@task
def research_task(self) -> Task:
    return Task(
        config=self.tasks_config['research_task'],
        output_pydantic=Company  # Structured output
    )
```

### Task Context (Dependencies)
```yaml
# In tasks.yaml
analysis_task:
  description: Analyze companies
  agent: analyst
  context:
    - research_task  # Uses output from research_task
```

---

## Next Steps

- See **hierarchical mode** section for advanced coordination
- See **LLM fallback strategy** for handling API failures
- See **my_stock_picker** example for a complete implementation
- See **[USING_MAX_RPM.md](../USING_MAX_RPM.md)** for rate limiting best practices

# Running in Hierarchical Mode: my_stock_picker Project

## What is Hierarchical Process?

In CrewAI, there are two main execution modes:
- **Sequential**: Tasks run one after another in order
- **Hierarchical**: A manager agent coordinates and delegates tasks to worker agents

## Key Components for Hierarchical Mode

### 1. Process Type
```python
process=Process.hierarchical  # Instead of Process.sequential
```

### 2. Manager Configuration (Two Options)

#### Option A: manager_agent (Recommended - More Control)
Create a full Agent with specific configuration:
```python
manager = Agent(
    config=self.agents_config['manager'],
    llm=self.llm_gemini,
    allow_delegation=True,  # Required for hierarchical
    max_rpm=8,  # Rate limit the manager too!
    verbose=True
)

return Crew(
    agents=self.agents,
    tasks=self.tasks,
    process=Process.hierarchical,
    manager_agent=manager,  # Pass the full agent
    verbose=True,
)
```

**Advantages:**
- Full control over manager's role, goal, and backstory (defined in YAML)
- Can customize manager's behavior and personality
- Better for complex workflows where manager needs specific expertise

**Important:** Manager agents also need rate limiting! They make many coordination requests.

#### Option B: manager_llm (Simpler)
Just provide an LLM, CrewAI creates a generic manager:
```python
return Crew(
    agents=self.agents,
    tasks=self.tasks,
    process=Process.hierarchical,
    manager_llm=self.llm_gemini,  # Just pass the LLM
    verbose=True,
)
```

**Advantages:**
- Simpler code
- CrewAI handles manager creation automatically
- Good for straightforward delegation workflows

**Disadvantages:**
- Less control over manager behavior
- Generic manager without specific domain expertise
- Can't set max_rpm directly (uses LLM's defaults)

## Rate Limiting in Hierarchical Mode

### Why Manager Needs Rate Limiting

In hierarchical mode, the manager makes **many more requests** than worker agents:
- Reads all task outputs
- Decides task assignments
- Coordinates between agents
- Monitors progress
- Synthesizes results

**Example request breakdown for a 3-task workflow:**
```
Manager requests: ~15-20 (coordination overhead)
Worker 1: ~5 requests
Worker 2: ~5 requests
Worker 3: ~5 requests
Total: ~30-35 requests
```

### Setting max_rpm on Manager

```python
@crew
def crew(self) -> Crew:
    manager = Agent(
        config=self.agents_config['manager'],
        llm=self.manager_llm,
        allow_delegation=True,
        # Conditional rate limiting
        max_rpm=8 if self.manager_llm == self.llm_gemini else None,
        verbose=True
    )

    return Crew(
        agents=self.agents,
        tasks=self.tasks,
        process=Process.hierarchical,
        manager_agent=manager,
        verbose=True
    )
```

### Rate Limit Strategy by Role

| Role | Request Volume | Recommended max_rpm |
|------|---------------|---------------------|
| **Manager** | High (coordination) | 8 for Gemini, 10 for Grok |
| **Researcher** | Medium-High (web searches) | 10 |
| **Analyst** | Medium (processing) | 10 |
| **Rater** | Low (scoring) | 10 |

### Avoiding Rate Limit Cascade

**Problem:** If manager hits rate limit, entire crew stalls.

**Solution 1:** Use local model for manager
```python
# Fast local coordination, no rate limits
manager = Agent(
    config=self.agents_config['manager'],
    llm=self.llm_ollama,  # Local, unlimited
    allow_delegation=True,
    verbose=True
)
```

**Solution 2:** Set conservative manager rate limit
```python
# Give manager 80% of API quota
manager = Agent(
    config=self.agents_config['manager'],
    llm=self.llm_gemini,
    allow_delegation=True,
    max_rpm=8,  # 80% of Gemini's 10 req/min
    verbose=True
)
```

**Solution 3:** Use different LLMs for manager vs workers
```python
# Manager: Fast Gemini (rate limited)
manager = Agent(llm=self.llm_gemini, max_rpm=8, ...)

# Workers: Unlimited local or higher-tier API
@agent
def researcher(self) -> Agent:
    return Agent(llm=self.llm_ollama, ...)  # No rate limit
```

## my_stock_picker Project Structure

### Agents (from agents.yaml)
1. **trending_company_finder**: Finds 2-3 trending publicly traded companies
2. **financial_researcher**: Researches and analyzes the companies
3. **stock_rater**: Rates companies on investment potential (1-100 scale)
4. **manager**: Coordinates all agents and delegates tasks

### Tasks (from tasks.yaml)
1. **find_trending_companies**: Search for trending stocks
   - Output: JSON list with ticker, price, reason
   
2. **research_trending_companies**: Deep analysis of found companies
   - Uses context from task 1
   - Output: Detailed research report
   
3. **rate_trending_companies**: Rate investment potential
   - Uses context from task 2
   - Output: Ranked list with ratings (1-100)

### Workflow
```
Manager (max_rpm=8)
  ├─> Delegates to trending_company_finder (max_rpm=10)
  │     └─> Completes find_trending_companies task
  │
  ├─> Delegates to financial_researcher (max_rpm=10)
  │     └─> Completes research_trending_companies task
  │
  └─> Delegates to stock_rater (max_rpm=10)
        └─> Completes rate_trending_companies task
```

## Implementation in my_stock_picker/src/my_stock_picker/crew.py

```python
@crew
def crew(self) -> Crew:
    """Creates the MyStockPicker crew"""

    # Create a manager agent with specific configuration
    manager = Agent(
        config=self.agents_config['manager'],
        llm=self.manager_llm,  # Configurable via llm_overrides
        allow_delegation=True,  # REQUIRED for hierarchical
        # Conditional rate limiting based on LLM
        max_rpm=8 if self.manager_llm == self.llm_gemini else None,
        verbose=True
    )

    return Crew(
        agents=self.agents,  # Worker agents (auto-created by @agent decorator)
        tasks=self.tasks,    # Tasks (auto-created by @task decorator)
        process=Process.hierarchical,  # KEY: Hierarchical mode
        manager_agent=manager,  # Pass the configured manager
        verbose=True,
    )
```

## Key Differences: Sequential vs Hierarchical

| Aspect | Sequential | Hierarchical |
|--------|-----------|-------------|
| **Execution** | Tasks run in order | Manager delegates tasks |
| **Coordination** | Implicit (task order) | Explicit (manager decides) |
| **Agent Assignment** | Tasks assigned to specific agents in YAML | Manager chooses which agent to assign |
| **Complexity** | Simpler | More complex |
| **Use Case** | Linear workflows | Complex coordination, parallel work |
| **Manager** | Not needed | Required (manager_agent or manager_llm) |
| **Rate Limiting** | Per-agent only | Manager + per-agent |
| **Request Volume** | Lower | Higher (manager overhead) |

## Rate Limiting Best Practices for Hierarchical

### 1. Always Rate Limit the Manager

```python
# ✅ Good: Manager has rate limit
manager = Agent(
    llm=self.llm_gemini,
    max_rpm=8,  # Protected from rate limit
    ...
)

# ❌ Bad: Manager has no rate limit
manager = Agent(
    llm=self.llm_gemini,
    # No max_rpm - will hit 429 errors!
    ...
)
```

### 2. Use Conditional Rate Limiting

```python
# Only limit API models, not local
max_rpm=8 if self.manager_llm == self.llm_gemini else None
```

### 3. Budget Your API Quota

Gemini free tier: 10 req/min total across ALL agents

**Bad distribution:**
```python
manager: max_rpm=10  # Uses all quota!
researcher: max_rpm=10  # Will fail!
```

**Good distribution:**
```python
manager: max_rpm=5    # 50% of quota
researcher: max_rpm=3  # 30% of quota
rater: max_rpm=2       # 20% of quota
# Total: 10 req/min (within limit)
```

### 4. Consider Hybrid Approach

```python
# Manager: Local (unlimited coordination)
manager = Agent(llm=self.llm_ollama, ...)

# Workers: API (quality when needed)
researcher = Agent(llm=self.llm_gemini, max_rpm=8, ...)
```

## Code Quality Notes

### Current Implementation
The current code is well-structured but could be improved:

1. **Manager LLM Choice**: Uses conditional `manager_llm` based on overrides
   - Good: Flexible and configurable
   - Has rate limiting: `max_rpm=8 if self.manager_llm == self.llm_gemini else None`

2. **Pydantic Models**: Good use of structured outputs
   - `TrendingCompanies` for task 1
   - `TrendingCompanyResearchList` for task 2
   - No Pydantic for task 3 (just markdown output)

3. **Tool Usage**: SerperDevTool assigned to agents that need web search
   - Good: Only finders have search tools
   - Note: Manager doesn't need tools (just delegates)

### Potential Improvements

1. **Add rate limit monitoring**:
```python
# Track rate limit hits
import logging
logger = logging.getLogger(__name__)

@crew
def crew(self) -> Crew:
    manager = Agent(
        ...,
        max_rpm=8,
        verbose=True  # Shows when rate limiting activates
    )
```

2. **Add memory to agents** for better context retention:
```python
@agent
def researcher(self) -> Agent:
    return Agent(
        config=self.agents_config['researcher'],
        llm=self.llm_grok,
        memory=True,  # Remember previous interactions
        max_rpm=10,
        verbose=True,
        tools=[SerperDevTool()]
    )
```

3. **Add validation to Pydantic models**:
```python
from pydantic import field_validator

class TrendingCompany(BaseModel):
    ticker: str
    
    @field_validator('ticker')
    def validate_ticker(cls, v):
        if not v or len(v) > 5:
            raise ValueError('Invalid ticker symbol')
        return v.upper()
```

## Running the Project

From `3_crew/my_stock_picker` directory:
```bash
crewai run
```

The crew will prompt for `sector` input (e.g., "technology", "healthcare").

## Debugging Rate Limits

Enable verbose mode to see rate limiting in action:

```bash
# Verbose output shows:
# - When max_rpm throttling happens
# - Which agent is waiting
# - How long until next request allowed
crewai run --verbose
```

Or in code:
```python
return Crew(
    agents=self.agents,
    tasks=self.tasks,
    verbose=True,  # Shows rate limit handling
    ...
)
```

## Summary

- ✅ Always set `max_rpm` on manager agents in hierarchical mode
- ✅ Budget API quota across all agents (don't exceed total limit)
- ✅ Use conditional rate limiting (`if llm == gemini`)
- ✅ Consider local models for high-request roles (manager)
- ✅ Enable verbose mode to monitor rate limiting
- ✅ See [USING_MAX_RPM.md](../USING_MAX_RPM.md) for comprehensive guide

# LLM Fallback Strategy

## The Problem

When running crews in production, you may encounter:
- **Rate limits**: "Too many requests" from Gemini/Grok
- **Service outages**: API temporarily unavailable
- **Quota exhaustion**: Daily/monthly limits exceeded

## The Solution: Fallback Chain

Implement a fallback chain that tries multiple LLMs in order:

```
Gemini (fast, cheap) → Grok (capable, moderate) → Ollama (local, always available)
```

## Implementation

### 1. Update Crew __init__ to Accept Override

```python
def __init__(self, manager_llm_override=None):
    """
    Args:
        manager_llm_override: Override which LLM the manager uses
                             Options: 'gemini-2.5-flash', 'grok-4-fast', 'ollama-gemma27b'
    """
    llms = build_llms(['grok-4-fast', 'gemini-2.5-flash', 'ollama-gemma27b'])
    self.llm_grok = llms['grok-4-fast']
    self.llm_gemini = llms['gemini-2.5-flash']
    self.llm_ollama = llms['ollama-gemma27b']
    
    # Support dynamic manager LLM selection
    if manager_llm_override:
        llm_map = {
            'gemini-2.5-flash': self.llm_gemini,
            'grok-4-fast': self.llm_grok,
            'ollama-gemma27b': self.llm_ollama
        }
        self.manager_llm = llm_map.get(manager_llm_override, self.llm_gemini)
    else:
        self.manager_llm = self.llm_gemini  # Default
```

### 2. Use manager_llm in Crew Definition

```python
@crew
def crew(self) -> Crew:
    manager = Agent(
        config=self.agents_config['manager'],
        llm=self.manager_llm,  # Dynamic LLM based on override
        allow_delegation=True,
        verbose=True
    )
    
    return Crew(
        agents=self.agents,
        tasks=self.tasks,
        process=Process.hierarchical,
        manager_agent=manager,
        verbose=True,
    )
```

### 3. Create Fallback Runner

File: `3_crew/crew_runner_with_fallback.py`

```python
def run_crew_with_fallback(crew_class, manager_llms, inputs=None, max_retries=3):
    """
    Run crew with automatic fallback if LLM fails.
    
    Args:
        crew_class: Crew class to instantiate
        manager_llms: List of LLM names to try ['gemini-2.5-flash', 'grok-4-fast', ...]
        inputs: Input dict for crew
        max_retries: Max retries per LLM
    
    Returns:
        CrewOutput if successful
    """
    for llm_name in manager_llms:
        for attempt in range(max_retries):
            try:
                crew_instance = crew_class(manager_llm_override=llm_name)
                result = crew_instance.crew().kickoff(inputs=inputs)
                return result  # Success!
            
            except Exception as e:
                # Check if rate limit error
                if 'rate limit' in str(e).lower() or '429' in str(e):
                    if attempt < max_retries - 1:
                        wait_time = 2 ** attempt  # Exponential backoff
                        time.sleep(wait_time)
                        continue
                    else:
                        break  # Move to next LLM
                else:
                    break  # Non-rate-limit error, move to next LLM
    
    raise Exception("All LLMs failed")
```

### 4. Usage Example

```python
from my_stock_picker.crew import MyStockPicker
from crew_runner_with_fallback import run_crew_with_fallback

result = run_crew_with_fallback(
    crew_class=MyStockPicker,
    manager_llms=['gemini-2.5-flash', 'grok-4-fast', 'ollama-gemma27b'],
    inputs={'sector': 'technology'},
    max_retries=2
)
```

## Execution Flow

```
1. Try Gemini with input
   └─ Rate limit → Retry (wait 1s)
      └─ Rate limit again → Move to next LLM

2. Try Grok with input
   └─ Success! → Return result

(If Grok also failed, would try Ollama)
```

## Best Practices

### 1. Order Your Fallback Chain Strategically

```python
# Fast → Capable → Always Available
['gemini-2.5-flash', 'grok-4-fast', 'ollama-gemma27b']
```

**Reasoning:**
- **Gemini**: Fastest and cheapest, try first
- **Grok**: More capable reasoning, good fallback
- **Ollama**: Local model, always works (no rate limits)

### 2. Implement Exponential Backoff

```python
wait_time = 2 ** attempt  # 1s, 2s, 4s, 8s...
```

Avoids hammering the API immediately after failure.

### 3. Detect Different Error Types

```python
error_msg = str(e).lower()

if any(keyword in error_msg for keyword in 
       ['rate limit', 'quota', 'overloaded', '429', 'too many requests']):
    # Retry or fallback
elif 'authentication' in error_msg or '401' in error_msg:
    # Don't retry, fix API key
elif 'timeout' in error_msg:
    # Network issue, maybe retry with longer timeout
```

### 4. Log Everything

```python
import logging

logger.info(f"Attempting crew with {llm_name}")
logger.warning(f"Rate limit on {llm_name}, attempt {attempt}")
logger.error(f"All LLMs failed. Last error: {e}")
```

Helps debug which LLM failed and why.

### 5. Consider Cost vs Reliability

| LLM | Speed | Cost | Reliability | When to Use |
|-----|-------|------|-------------|-------------|
| Gemini Flash | ⚡⚡⚡ | 💰 | 🔄 (rate limits) | First choice, development |
| Grok | ⚡⚡ | 💰💰 | 🔄🔄 (fewer limits) | Production fallback |
| Ollama | ⚡ | Free | ✅ Always works | Final fallback, local dev |

## Alternative: Worker Agent Fallbacks

You could also apply fallbacks to individual worker agents:

```python
@agent
def researcher(self) -> Agent:
    # Try Grok first for research, fallback to Gemini
    llm = self.llm_grok if not hasattr(self, '_research_failed') else self.llm_gemini
    
    return Agent(
        config=self.agents_config['researcher'],
        llm=llm,
        verbose=True,
        tools=[SerperDevTool()]
    )
```

But manager fallback is usually sufficient since the manager coordinates everything.

## Files Created

1. **3_crew/crew_runner_with_fallback.py**: Reusable fallback runner
2. **my_stock_picker/run_with_fallback.py**: Example usage script
3. **my_stock_picker/src/my_stock_picker/crew.py**: Updated with manager_llm_override support

# Memory Types in CrewAI

## Overview

CrewAI provides five types of memory to help agents remember context, learn from past executions, and maintain information about entities. Memory can be enabled at both the **agent level** and **crew level**.

## Memory Types

### 1. Short-Term Memory (STM)
**Stores**: Recent interactions and context from the **current execution**

**How it works:**
- Uses **RAG (Retrieval-Augmented Generation)** with embeddings
- Enables semantic search (search by meaning, not just keywords)
- Cleared between major runs (not persistent like LTM)
- Stored in vector database for fast retrieval

**Use case:**
- Agent needs to remember what another agent said earlier in THIS run
- Maintaining context throughout a multi-step workflow
- Recalling specific details from earlier tasks

**Example:**
```python
short_term_memory = ShortTermMemory(
    storage=RAGStorage(
        embedder_config={
            "provider": "openai",
            "config": {"model": "text-embedding-3-small"}
        },
        type="short_term",
        path="./memory/"
    )
)
```

**Storage:** Vector embeddings in `./memory/short_term/`

---

### 2. Long-Term Memory (LTM)
**Stores**: Valuable insights and learnings **across multiple executions**

**How it works:**
- Persists information permanently in a database (usually SQLite)
- Survives between runs, days, weeks, months
- Agent can learn from past experiences
- Builds knowledge over time

**Use case:**
- Remembering companies you've already analyzed
- Avoiding duplicate recommendations
- Learning from past successful/failed decisions
- Building historical context

**Example:**
```python
long_term_memory = LongTermMemory(
    storage=LTMSQLiteStorage(
        db_path="./memory/long_term_memory_storage.db"
    )
)
```

**Storage:** SQLite database `./memory/long_term_memory_storage.db`

---

### 3. Entity Memory
**Stores**: Information about people, places, companies, and concepts

**How it works:**
- Tracks specific entities encountered during tasks
- Uses RAG for semantic storage and retrieval
- Builds relationships between entities
- Enables deeper understanding of entity properties

**Use case:**
- Tracking companies: "Apple", "Tesla", "Microsoft"
- Remembering facts: "Apple has high P/E ratio"
- Linking entities: "Apple is a competitor of Microsoft"
- Understanding relationships

**Example:**
```python
entity_memory = EntityMemory(
    storage=RAGStorage(
        embedder_config={
            "provider": "openai",
            "config": {"model": "text-embedding-3-small"}
        },
        type="short_term",
        path="./memory/"
    )
)
```

**Storage:** Vector embeddings in `./memory/entity/`

---

### 4. Contextual Memory
**Stores**: Combined context from all other memory types

**How it works:**
- Automatically combines Short-Term, Long-Term, and Entity Memory
- Provides holistic view of all available context
- Agent queries all memory types simultaneously
- Most comprehensive memory option

**Use case:**
- Agent needs full picture (recent context + historical + entity info)
- Complex decision-making requiring multiple memory types
- Ensuring no relevant information is missed

**Implementation:**
Enabled automatically when you set `memory=True` at crew or agent level.

---

### 5. User Memory
**Stores**: User-specific information and preferences

**How it works:**
- Stored separately per user
- Requires manual management and inclusion in prompts
- Not automatically retrieved
- Developer controls what gets stored/retrieved

**Use case:**
- User preferences: "prefers tech stocks", "risk-averse investor"
- Personalization: "always exclude tobacco companies"
- User history: "invested in AAPL on 2025-01-10"

**Implementation:**
Custom - you manage storage and retrieval in your code.

---

## Enabling Memory

### Option 1: Agent-Level Memory

Enable memory for specific agents:

```python
@agent
def trending_company_finder(self) -> Agent:
    return Agent(
        config=self.agents_config['trending_company_finder'],
        tools=[SerperDevTool()],
        memory=True  # Enable memory for this agent
    )
```

**When to use:**
- Only certain agents need memory
- Different agents need different memory capabilities
- Fine-grained control

### Option 2: Crew-Level Memory

Enable memory for the entire crew:

```python
@crew
def crew(self) -> Crew:
    return Crew(
        agents=self.agents,
        tasks=self.tasks,
        process=Process.hierarchical,
        memory=True,  # Enable memory for all agents
        long_term_memory=LongTermMemory(...),
        short_term_memory=ShortTermMemory(...),
        entity_memory=EntityMemory(...),
    )
```

**When to use:**
- All agents should share memory
- Consistent memory across workflow
- Simpler configuration

---

## Real-World Example: Stock Picker Crew

From `3_crew/stock_picker/src/stock_picker/crew.py`:

### Agents with Memory

```python
@agent
def trending_company_finder(self) -> Agent:
    return Agent(
        config=self.agents_config['trending_company_finder'],
        tools=[SerperDevTool()],
        memory=True  # ✅ Needs to remember past picks
    )

@agent
def financial_researcher(self) -> Agent:
    return Agent(
        config=self.agents_config['financial_researcher'],
        tools=[SerperDevTool()]
        # ❌ No memory - just analyzes what's given
    )

@agent
def stock_picker(self) -> Agent:
    return Agent(
        config=self.agents_config['stock_picker'],
        tools=[PushNotificationTool()],
        memory=True  # ✅ Needs to remember past selections
    )
```

### Crew-Level Memory Configuration

```python
@crew
def crew(self) -> Crew:
    return Crew(
        agents=self.agents,
        tasks=self.tasks,
        process=Process.hierarchical,
        memory=True,  # Enable crew-wide memory
        
        # Long-term: Persistent across runs
        long_term_memory=LongTermMemory(
            storage=LTMSQLiteStorage(
                db_path="./memory/long_term_memory_storage.db"
            )
        ),
        
        # Short-term: Current run context
        short_term_memory=ShortTermMemory(
            storage=RAGStorage(
                embedder_config={
                    "provider": "openai",
                    "config": {"model": "text-embedding-3-small"}
                },
                type="short_term",
                path="./memory/"
            )
        ),
        
        # Entity: Track companies, people, concepts
        entity_memory=EntityMemory(
            storage=RAGStorage(
                embedder_config={
                    "provider": "openai",
                    "config": {"model": "text-embedding-3-small"}
                },
                type="short_term",
                path="./memory/"
            )
        ),
    )
```

### How It Works in Practice

**Goal**: "Always pick new companies. Don't pick the same company twice."

**Run 1 (Jan 13):**
1. `trending_company_finder` finds: Apple, Tesla, NVIDIA
2. **LTM stores**: "Found Apple, Tesla, NVIDIA on 2025-01-13"
3. `stock_picker` selects: Apple
4. **LTM stores**: "Picked Apple on 2025-01-13"

**Run 2 (Jan 14):**
1. `trending_company_finder` checks **LTM**: "What companies did I find before?"
2. **Avoids**: Apple, Tesla, NVIDIA (already found)
3. Finds **new**: Microsoft, Google, Meta
4. `stock_picker` checks **LTM**: "What did I pick before?"
5. **Avoids**: Apple (already picked)
6. Picks: Microsoft
7. **LTM stores**: New findings and pick

**Run 3 (Jan 15):**
1. Both agents check **LTM**
2. **Avoid**: Apple, Tesla, NVIDIA, Microsoft, Google, Meta
3. Find/pick only **NEW** companies

---

## Memory Storage

Memory is stored in the `./memory/` directory:

```
memory/
├── long_term_memory_storage.db    # SQLite (persistent)
├── short_term/                    # RAG embeddings (current run)
│   ├── chroma.sqlite3
│   └── embeddings/
└── entity/                        # Entity tracking
    ├── chroma.sqlite3
    └── embeddings/
```

---

## Cost Considerations

⚠️ **Memory with RAG uses embeddings**, which costs money:

| Memory Type | Storage | Embeddings | Cost |
|-------------|---------|------------|------|
| Long-Term | SQLite | ❌ No | Free |
| Short-Term | RAG | ✅ Yes | $$ |
| Entity | RAG | ✅ Yes | $$ |

**Embedding costs:**
- OpenAI `text-embedding-3-small`: $0.02 per 1M tokens
- Each memory store/retrieve creates embeddings
- Can add up with frequent operations

**Cost reduction strategies:**

1. **Use only Long-Term Memory** (no RAG):
```python
memory=True,
long_term_memory=LongTermMemory(...)
# Don't configure short_term_memory or entity_memory
```

2. **Use local embeddings** (free, but slower):
```python
embedder_config={
    "provider": "huggingface",  # Free local model
    "config": {"model": "sentence-transformers/all-MiniLM-L6-v2"}
}
```

3. **Disable memory** if not needed:
```python
# Just don't set memory=True
```

---

## When to Use Which Memory Type

| Scenario | Use This Memory |
|----------|----------------|
| Avoid duplicate recommendations | Long-Term |
| Maintain context within a run | Short-Term |
| Track company facts and relationships | Entity |
| Agent needs full context | Contextual (all) |
| User preferences | User (custom) |
| Simple workflows, no history | None |

---

## Best Practices

1. **Start simple**: Enable only what you need
   - Most crews don't need all memory types
   - Long-Term is often sufficient

2. **Memory at agent vs crew level**:
   - Agent-level: Fine-grained control, specific needs
   - Crew-level: Simpler, shared memory

3. **Choose the right storage**:
   - SQLite for structured, queryable data
   - RAG for semantic search capabilities

4. **Monitor costs**:
   - RAG-based memory uses OpenAI embeddings
   - Can add up with frequent operations

5. **Clear memory when appropriate**:
   - Delete `./memory/` directory to reset
   - Useful for testing or starting fresh

---

## Common Patterns

### Pattern 1: "No Duplicates" (Stock Picker)
```python
# Agents with memory check LTM before acting
memory=True
long_term_memory=LongTermMemory(...)
```

### Pattern 2: "Conversation Flow" (Chatbot)
```python
# Maintain context throughout conversation
memory=True
short_term_memory=ShortTermMemory(...)
```

### Pattern 3: "Learn from History" (Trading Bot)
```python
# Learn from past trades
memory=True
long_term_memory=LongTermMemory(...)
entity_memory=EntityMemory(...)  # Track stocks
```

### Pattern 4: "No Memory Needed" (One-off Analysis)
```python
# Simple, stateless workflow
# Don't set memory=True
```

# Using FAISS for Local Memory Storage (Cost-Free Alternative)

## The Problem with Default RAG Storage

CrewAI's default RAG storage uses **OpenAI embeddings** which cost money:
- OpenAI `text-embedding-3-small`: $0.02 per 1M tokens
- Every memory store/retrieve creates embeddings
- Can add up quickly with frequent operations

## The Solution: FAISS with Local Embeddings

**FAISS** (Facebook AI Similarity Search) is a library for efficient similarity search with **local, free embeddings**.

### What is FAISS?

- **Vector database** optimized for similarity search
- Runs **completely locally** (no API calls)
- Uses **CPU or GPU** for fast vector operations
- Stores embeddings in memory or on disk
- **Free and open-source**

### Architecture

```
┌─────────────────────────────────────────────────────────┐
│  Agent Memory Request                                   │
└────────────────┬────────────────────────────────────────┘
                 │
                 ▼
┌─────────────────────────────────────────────────────────┐
│  FAISSStorage (Custom Implementation)                   │
│  ├─ Uses HuggingFace sentence-transformers             │
│  ├─ Creates embeddings locally (FREE)                   │
│  └─ Stores in FAISS index                               │
└────────────────┬────────────────────────────────────────┘
                 │
                 ▼
┌─────────────────────────────────────────────────────────┐
│  FAISS Index (Vector Database)                          │
│  ├─ IndexFlatL2: Exact similarity search                │
│  ├─ InMemoryDocstore: Text storage                      │
│  └─ Saves to disk: ./memory/short_term_faiss/           │
└─────────────────────────────────────────────────────────┘
```

## Required Imports

```python
# CrewAI Memory
from crewai.memory import LongTermMemory, ShortTermMemory, EntityMemory
from crewai.memory.storage.rag_storage import RAGStorage
from crewai.memory.storage.ltm_sqlite_storage import LTMSQLiteStorage

# FAISS and Vector Storage
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings
import faiss
```

### Required Dependencies

Add to `pyproject.toml`:

```toml
[project]
dependencies = [
    "crewai[google-genai,tools]==1.4.1",
    "faiss-cpu>=1.12.0",              # FAISS for vector search
    "langchain>=1.0.5",                # LangChain base
    "langchain-community>=0.4.1",      # FAISS integration
    "langchain-core>=1.0.4",           # Core LangChain
    "langchain-huggingface>=1.0.1",    # HuggingFace embeddings
    "sentence-transformers>=5.1.2",    # Local embedding models
]
```

## Custom FAISSStorage Implementation

```python
class FAISSStorage(RAGStorage):
    """
    Custom FAISS-based storage that uses local embeddings instead of OpenAI.
    
    Benefits:
    - 100% free (no API costs)
    - Runs locally (no network calls)
    - Fast similarity search
    - Persistent storage
    """
    
    def __init__(self, embedder_config, type, path):
        # Skip super().__init__ to avoid CrewAI's default OpenAI embedder
        self.embedder_config = embedder_config
        self.type = type
        self.path = path
        
        # Create local HuggingFace embeddings (FREE!)
        self.embedding_function = HuggingFaceEmbeddings(
            model_name=embedder_config['config']['model']
        )
        
        # Load existing FAISS index or create new one
        if os.path.exists(path):
            # Load from disk
            self.vectorstore = FAISS.load_local(
                path, 
                self.embedding_function, 
                allow_dangerous_deserialization=True
            )
        else:
            # Create new empty index
            # Get embedding dimension by embedding a dummy text
            dimension = len(self.embedding_function.embed_query("dummy text"))
            
            # Create FAISS index (L2 distance for similarity)
            index = faiss.IndexFlatL2(dimension)
            
            # Create FAISS vectorstore
            self.vectorstore = FAISS(
                embedding_function=self.embedding_function.embed_query,
                index=index,
                docstore=InMemoryDocstore({}),
                index_to_docstore_id={},
            )
        
        # Set collection name
        self.collection_name = f"{type}_memory"

    def search(self, query, **kwargs):
        """Search for similar documents"""
        return self.vectorstore.similarity_search(query, **kwargs)
    
    def add(self, texts, metadatas=None, **kwargs):
        """Add new documents and save to disk"""
        self.vectorstore.add_texts(texts, metadatas, **kwargs)
        self.vectorstore.save_local(self.path)  # Persist to disk
```

## Using FAISS in Your Crew

### Configure Memory with FAISS

```python
@crew
def crew(self) -> Crew:
    return Crew(
        agents=self.agents,
        tasks=self.tasks,
        process=Process.hierarchical,
        manager_agent=manager,
        memory=True,
        
        # Long-term: SQLite (free, no embeddings needed)
        long_term_memory=LongTermMemory(
            storage=LTMSQLiteStorage(
                db_path="./memory/long_term_memory_storage.db"
            )
        ),
        
        # Short-term: FAISS with local embeddings (FREE!)
        short_term_memory=ShortTermMemory(
            storage=FAISSStorage(
                embedder_config={
                    "provider": "huggingface",
                    "config": {
                        "model": 'sentence-transformers/all-MiniLM-L6-v2'
                    }
                },
                type="short_term",
                path="./memory/short_term_faiss"
            )
        ),
        
        # Entity: FAISS with local embeddings (FREE!)
        entity_memory=EntityMemory(
            storage=FAISSStorage(
                embedder_config={
                    "provider": "huggingface",
                    "config": {
                        "model": 'sentence-transformers/all-MiniLM-L6-v2'
                    }
                },
                type="entity",
                path="./memory/entity_faiss"
            )
        ),
    )
```

## Local Embedding Models

### Recommended Models

| Model | Size | Dimensions | Speed | Use Case |
|-------|------|------------|-------|----------|
| **all-MiniLM-L6-v2** | 80MB | 384 | ⚡⚡⚡ Fast | General purpose, good quality |
| all-mpnet-base-v2 | 420MB | 768 | ⚡⚡ Medium | Higher quality, slower |
| all-MiniLM-L12-v2 | 120MB | 384 | ⚡⚡ Medium | Balance of speed/quality |
| paraphrase-MiniLM-L3-v2 | 61MB | 384 | ⚡⚡⚡ Very Fast | Fastest, lower quality |

**Recommendation:** `sentence-transformers/all-MiniLM-L6-v2` (best balance)

### First Run: Model Download

The first time you run, the model downloads automatically:

```bash
# First run downloads model (~80MB for all-MiniLM-L6-v2)
crewai run
# Downloading model: sentence-transformers/all-MiniLM-L6-v2
# Downloaded to: ~/.cache/huggingface/

# Subsequent runs use cached model (instant)
crewai run
```

Models are cached in `~/.cache/huggingface/` and reused.

## Storage Structure

```
memory/
├── long_term_memory_storage.db    # SQLite (Long-term)
├── short_term_faiss/               # FAISS Short-term
│   ├── index.faiss                 # Vector index
│   └── index.pkl                   # Docstore + metadata
└── entity_faiss/                   # FAISS Entity
    ├── index.faiss                 # Vector index
    └── index.pkl                   # Docstore + metadata
```

## How FAISS Works

### 1. Embedding Creation

```python
# Text to embed
text = "Apple announced new iPhone with AI features"

# Local embedding (FREE, ~10ms)
embedding = embedding_function.embed_query(text)
# Result: [0.123, -0.456, 0.789, ...] (384 dimensions)
```

### 2. Vector Storage

```python
# Add to FAISS index
index.add(embedding)  # Store in L2 index
docstore[id] = text   # Store original text
```

### 3. Similarity Search

```python
# Query
query = "What did Apple announce?"
query_embedding = embedding_function.embed_query(query)

# FAISS finds nearest neighbors (cosine/L2 distance)
similar_docs = index.search(query_embedding, k=3)
# Returns: Top 3 most similar documents
```

### L2 Distance vs Cosine Similarity

**IndexFlatL2** uses L2 (Euclidean) distance:
```
distance = sqrt((v1[0]-v2[0])² + (v1[1]-v2[1])² + ...)
```

- Smaller distance = more similar
- Fast exact search
- Good for normalized embeddings (sentence-transformers outputs are normalized)

## Cost Comparison

### OpenAI Embeddings (Default RAG)

```
Cost per embedding: $0.00002 per 1K tokens
Average memory operation: ~500 tokens
Cost per operation: ~$0.00001

Daily crew runs: 10 runs × 50 operations = 500 operations
Daily cost: 500 × $0.00001 = $0.005/day
Monthly cost: ~$0.15/month
```

### FAISS with Local Embeddings

```
Cost per embedding: $0 (local)
Cost per operation: $0 (local)

Daily crew runs: Unlimited
Daily cost: $0
Monthly cost: $0
```

**Savings:** 100% cost reduction for memory operations!

## Performance Considerations

### Speed Comparison

| Operation | OpenAI | FAISS Local |
|-----------|--------|-------------|
| **Embed text** | ~200ms (API) | ~10ms (local) |
| **Search** | ~200ms (API) | <1ms (local) |
| **Store** | ~200ms (API) | ~5ms (local) |

**FAISS is 20-200x faster** because it's local!

### Memory Usage

```
Model loaded in RAM: ~100MB (all-MiniLM-L6-v2)
FAISS index: ~1KB per document
1000 documents: ~1.1MB total memory

Negligible for modern systems.
```

### Disk Usage

```
Model cache: ~80MB (one-time download)
FAISS indices: ~1KB per document
1000 documents: ~1MB on disk

Very efficient!
```

## Advantages of FAISS

### ✅ Pros

1. **Free** - No API costs
2. **Fast** - Local computation (20-200x faster than API)
3. **Private** - Data never leaves your machine
4. **Offline** - Works without internet
5. **Scalable** - Handles millions of vectors efficiently
6. **Persistent** - Saves to disk automatically

### ❌ Cons

1. **Initial download** - First run downloads model (~80MB)
2. **RAM usage** - Model stays in memory (~100MB)
3. **Quality** - Slightly lower quality than OpenAI embeddings
   - OpenAI: ~95% accuracy
   - all-MiniLM-L6-v2: ~90% accuracy
   - Difference is minor for most use cases

## When to Use FAISS vs OpenAI

### Use FAISS When:

- ✅ Cost is a concern (free tier, hobby projects)
- ✅ Speed matters (low latency)
- ✅ Privacy is important (data stays local)
- ✅ Offline operation needed
- ✅ High volume of operations

### Use OpenAI When:

- ✅ Maximum quality required (production, critical decisions)
- ✅ Don't want to manage model downloads
- ✅ Cost is not a constraint
- ✅ Want latest embedding improvements

## Migration: OpenAI → FAISS

### Before (OpenAI RAG)

```python
short_term_memory=ShortTermMemory(
    storage=RAGStorage(
        embedder_config={
            "provider": "openai",  # Costs money!
            "config": {"model": "text-embedding-3-small"}
        },
        type="short_term",
        path="./memory/"
    )
)
```

### After (FAISS Local)

```python
short_term_memory=ShortTermMemory(
    storage=FAISSStorage(
        embedder_config={
            "provider": "huggingface",  # Free!
            "config": {"model": "sentence-transformers/all-MiniLM-L6-v2"}
        },
        type="short_term",
        path="./memory/short_term_faiss"
    )
)
```

**That's it!** The rest of your crew code stays the same.

## Troubleshooting

### Issue: Model Download Fails

```bash
# Manual download
python -c "from sentence_transformers import SentenceTransformer; SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')"
```

### Issue: ImportError for InMemoryDocstore

```python
# Old import (doesn't work):
from langchain.docstore import InMemoryDocstore

# New import (works):
from langchain_community.docstore.in_memory import InMemoryDocstore
```

### Issue: "embedding_function is expected to be an Embeddings object"

This is a warning, not an error. You can ignore it or fix it:

```python
# Instead of:
embedding_function=self.embedding_function.embed_query

# Use:
embedding_function=self.embedding_function  # Pass the object, not method
```

## Summary

- ✅ **FAISS = Free, fast, local vector database**
- ✅ **HuggingFace = Free, local embedding models**
- ✅ **Replace OpenAI RAG with FAISSStorage**
- ✅ **100% cost reduction for memory operations**
- ✅ **20-200x faster than API-based embeddings**
- ✅ **Works offline, private, scalable**
- ✅ **Only ~100MB RAM overhead**

**Perfect for:**
- Development and testing
- Hobby projects
- Cost-sensitive production
- High-volume operations
- Privacy-focused applications